# Player Level & Game Day Progression Analysis

**Purpose:** Understand how quickly players progress through levels and game days across install cohorts, and how retention curves compare over time.

**Data range:** April 2024 – April 2026 install cohorts (~59M player-day observations)

---

## Key Findings

- **Level progression is rapid in the first few days:** Players reach an average of ~4 levels on install day (D0), ~6 by D1, ~7 by D2, and ~8 by D3.
- **Recent cohorts show slightly lower early engagement:** Average max level on D0 dropped from ~4.0 (Apr 2024) to ~3.5 (Apr 2026) — roughly an 11% decrease.
- **D1 retention has declined slightly:** ~55% of the Apr 2024 cohort returned on D1 vs ~51% for Apr 2026, suggesting a softening in early-day engagement over time.
- **Wide player distribution:** P90 players progress ~3–4× faster than P10, and this spread widens significantly beyond D7.
- **Game day and level are tightly coupled:** Game day progression closely mirrors level progression (~2 game days on D0, ~3.5 on D1), indicating most sessions drive both metrics in tandem.

In [2]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np

## Get data

### Player level and game day

In [ ]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/playerlevel.sql'
parameters = {
    'start_date': '2024-04-01',
}

bqc = BigQueryConnector()
# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

In [ ]:
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache
refresh_data = False
if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [92]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday
0,644C7D9B02037937,2024-08-18,2024-08-18,2024-08-18,2024-08-01,0,6,4
1,6B93162278259089,2024-05-16,2024-05-14,2024-05-12,2024-05-01,2,8,5
2,6CCE7309BA30BA50,2024-07-22,2024-07-21,2024-07-21,2024-07-01,1,6,4
3,FA0F5A38E6F51368,2025-06-07,2025-05-30,2025-05-25,2025-05-01,8,8,5
4,E55CE0D7877039F4,2025-07-22,2024-05-06,2024-05-05,2024-05-01,442,138,177
...,...,...,...,...,...,...,...,...
59458542,8AAE88246D01A515,2025-01-29,2024-10-03,2024-09-29,2024-10-01,118,58,65
59458543,A2BB0217A6076FAC,2025-01-11,2024-10-04,2024-09-29,2024-10-01,99,13,9
59458544,C517F33739B40992,2024-10-16,2024-09-20,2024-09-15,2024-09-01,26,19,14
59458545,E971FF9009EC9ED0,2024-07-17,2024-04-27,2024-04-21,2024-04-01,81,24,20


### Bootstrap

In [37]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/bootstrap.sql'
parameters = {
    'start_date': '2026-04-01',
}

bqc = BigQueryConnector()
# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 13.81 GB when run.
Estimated query cost: $0.09


In [ ]:
data_bootstrap = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache
refresh_data = False
if refresh_data:
    data_bootstrap = bqc.get(query='./sql/bootstrap.sql', is_path=True, query_parameters=parameters)
    data_bootstrap.to_pickle('./data/bootstrap.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data_bootstrap = pd.read_pickle('./data/bootstrap.pkl')

In [39]:
data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence
0,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.782472Z,2026-04-02,10003F6FA8F5957E,app_launch,cmpt,0.355,False,1
1,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.783575Z,2026-04-02,10003F6FA8F5957E,display_loading_screen,strt,0.389,False,2
2,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.784469Z,2026-04-02,10003F6FA8F5957E,service_manager_process,strt,0.406,False,3
3,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.785349Z,2026-04-02,10003F6FA8F5957E,service_manager_process,cmpt,0.798,False,4
4,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.786193Z,2026-04-02,10003F6FA8F5957E,auth_process,strt,0.806,False,5
...,...,...,...,...,...,...,...,...,...
109731916,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.558424Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,strt,15.502,True,3
109731917,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.560164Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,cmpt,15.504,True,4
109731918,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:45.940106Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,strt,20.883,True,5
109731919,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:46.152842Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,cmpt,21.096,True,6


## Process data

In [ ]:
dt_mode = 'install_dt_month'

data['install_dt'] = data[dt_mode]

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]
data

In [ ]:
# Cache the filtered dataset to avoid reprocessing on subsequent runs
data.to_pickle('./data/playprogression_processed.pkl')

In [ ]:
# Load the cohort-filtered processed dataset
data = pd.read_pickle('./data/playprogression_processed.pkl') 

## Player Level reached at day x

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [ ]:
# Step 1: Count unique users per (install cohort, day since install, max_level bucket)
player_level_agg = data.groupby(['install_dt', 'days_since_install', 'max_level']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day (denominator for percentage share)
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Compute the share of each day's users sitting at each level
player_level_agg = player_level_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
player_level_agg['percentage_of_daily_users'] = player_level_agg['unique_users'] / player_level_agg['total_unique_users']

# Step 4: Weighted average max level per cohort-day (users as weights)
weighted_avg = player_level_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_level'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_level']

player_level_agg = player_level_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small level buckets (< 50 users) that could distort the weighted average
player_level_agg = player_level_agg[player_level_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
player_level_agg = player_level_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_level = ('weighted_avg_max_level', 'first')
).reset_index()

player_level_agg

In [97]:
# hide-output
fig = px.line(player_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='install_dt',
              title='Player level daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [ ]:
def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    """Return P10 / P50 / P90 of measure_col, using user counts as weights.

    Sorts by the measure, accumulates weights, then uses searchsorted to find
    the value at each quantile threshold — equivalent to a weighted percentile.
    """
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)

In [ ]:
# hide-output
level_dist = data.groupby(['days_since_install', 'max_level']).agg(
    users=('user_id', 'count')
).reset_index()



level_pcts = level_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_level', include_groups=False).reset_index()

fig = px.line(
    level_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_level'),
    x='days_since_install',
    y='max_level',
    color='percentile',
    title='Player level distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [ ]:
player_level_agg2 = player_level_agg[['install_dt', 'days_since_install','weighted_avg_max_level']].drop_duplicates()

# Day-over-day % change in weighted avg level within each install cohort.
# Day 0 is filled as 1.0 (100%) since there is no prior day to compare against.
player_level_agg2['pct_change_weighted_avg_max_level'] = player_level_agg2.groupby('install_dt')['weighted_avg_max_level'].pct_change()
player_level_agg2.pct_change_weighted_avg_max_level = player_level_agg2.pct_change_weighted_avg_max_level.fillna(1)
player_level_agg2

In [99]:
# hide-output
fig = px.line(player_level_agg2.where(player_level_agg2.days_since_install<=30), 
              x='days_since_install', 
              y='pct_change_weighted_avg_max_level',
              color='install_dt',
              title='Player level daily progression by cohort (pct change)',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True})


fig.show()

## Game day reached at day x

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [ ]:
# Same weighted-average approach as player level, applied to max_gameday

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

In [104]:
# hide-output
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [108]:
# hide-output
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

## % of cohort users active by days since install

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [ ]:
# Count unique users active on each day since install, per cohort
users_agg = pd.DataFrame()
users_agg = data.groupby(['install_dt', 'days_since_install']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Total unique users ever observed in each cohort (denominator for retention %)
users_by_cohort = pd.DataFrame()
users_by_cohort = data.groupby('install_dt').agg(
    total_cohort_users = ('user_id', 'nunique')
).reset_index()

users_agg = users_agg.merge(users_by_cohort, on='install_dt')

# Retention rate: share of cohort still active on a given day
users_agg['pct_cohort_users_active'] = users_agg['unique_users'] / users_agg['total_cohort_users']

users_agg

In [110]:
# hide-output

# apply a log transformation to the y axis to better visualize the differences between cohorts, especially in the later days since install where the percentage of active users is very low
fig = px.line(users_agg, 
              x='days_since_install', 
              y='pct_cohort_users_active',
              color='install_dt',
              title='% of cohort users active by days since install (log scale)',
              width=1200,
              height=600,
              hover_data={'pct_cohort_users_active': ':.2%', 'unique_users': True, 'total_cohort_users': True},
              log_y=True)


fig.show()

# Things to do next
- Is the churn increasing for P90 players when they reach the 150 GD mark?
- Is the slowdown on pace seen on around level 50 on newest cohort caused by CPE mix? what does it look like with only organics?

# Bootstrap event analysis

_Section in progress._ This section will identify the earliest in-game milestone that best predicts long-term retention — the "bootstrap moment" — by comparing event completion rates between retained and churned players.

In [29]:
data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence
0,d8c7f9ac-e697-43d2-a0ee-58b14dbc2804,2026-04-01T06:47:36.223267Z,2026-04-01,1000E5F0A7C92171,app_launch,cmpt,0.335,False,1
1,d8c7f9ac-e697-43d2-a0ee-58b14dbc2804,2026-04-01T06:47:36.224145Z,2026-04-01,1000E5F0A7C92171,display_loading_screen,strt,0.363,False,2
2,d8c7f9ac-e697-43d2-a0ee-58b14dbc2804,2026-04-01T06:47:36.224871Z,2026-04-01,1000E5F0A7C92171,service_manager_process,strt,0.375,False,3
3,d8c7f9ac-e697-43d2-a0ee-58b14dbc2804,2026-04-01T06:47:36.225501Z,2026-04-01,1000E5F0A7C92171,service_manager_process,cmpt,0.656,False,4
4,d8c7f9ac-e697-43d2-a0ee-58b14dbc2804,2026-04-01T06:47:36.226115Z,2026-04-01,1000E5F0A7C92171,auth_process,strt,0.664,False,5
...,...,...,...,...,...,...,...,...,...
3277546,9a8a6fa0-d36c-47b6-8823-81fecec3349a,2026-04-01T23:56:47.995114Z,2026-04-01,FFFDA5046DDD73A7,auth_process,strt,1.439,False,5
3277547,9a8a6fa0-d36c-47b6-8823-81fecec3349a,2026-04-01T23:56:52.440053Z,2026-04-01,FFFDA5046DDD73A7,auth_process,cmpt,6.314,False,6
3277548,9a8a6fa0-d36c-47b6-8823-81fecec3349a,2026-04-01T23:56:52.4761Z,2026-04-01,FFFDA5046DDD73A7,helpshift_sdk,strt,6.350,False,7
3277549,9a8a6fa0-d36c-47b6-8823-81fecec3349a,2026-04-01T23:56:52.483273Z,2026-04-01,FFFDA5046DDD73A7,helpshift_sdk,cmpt,6.358,False,8


In [48]:
data_bootstrap = data_bootstrap[(data_bootstrap.context == 'cmpt')&(data_bootstrap.payload_date < pd.to_datetime('today'))]

data_bootstrap

,session_id,payload_timestamp,payload_date,user_id,tutorial_step_id,context,seconds,is_first_session,sequence
0,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.782472Z,2026-04-02,10003F6FA8F5957E,app_launch,cmpt,0.355,False,1
3,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:10.785349Z,2026-04-02,10003F6FA8F5957E,service_manager_process,cmpt,0.798,False,4
5,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:14.801886Z,2026-04-02,10003F6FA8F5957E,auth_process,cmpt,4.864,False,6
7,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:14.805722Z,2026-04-02,10003F6FA8F5957E,helpshift_sdk,cmpt,4.868,False,8
8,08b82d5d-5caf-4f22-97ab-ce23b0e8cd0a,2026-04-02T05:25:29.87858Z,2026-04-02,10003F6FA8F5957E,display_loading_screen,cmpt,19.937,False,9
...,...,...,...,...,...,...,...,...,...
109731913,None,2026-04-17T15:45:39.192703Z,2026-04-17,FFFFF67B4ED7017A,service_manager_process,cmpt,14.127,True,186380
109731915,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.555821Z,2026-04-17,FFFFF67B4ED7017A,auth_process,cmpt,15.499,True,2
109731917,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:40.560164Z,2026-04-17,FFFFF67B4ED7017A,helpshift_sdk,cmpt,15.504,True,4
109731919,e4f7e57b-44d8-4bc1-aac5-5f93bcd0cbc6,2026-04-17T15:45:46.152842Z,2026-04-17,FFFFF67B4ED7017A,external_notifications_popup,cmpt,21.096,True,6


In [49]:
# calculate the percentiles P90, P50 and P10 for the seconds metric by payload_date and tutorial_step_id
percentiles = data_bootstrap.groupby(['payload_date', 'tutorial_step_id'])['seconds'].quantile([0.1, 0.5, 0.9]).unstack()
percentiles.columns = ['P10', 'P50', 'P90']
percentiles.reset_index(inplace=True)

percentiles_melted = percentiles.melt(id_vars=['payload_date', 'tutorial_step_id'], var_name='percentile', value_name='seconds').sort_values(['payload_date', 'tutorial_step_id', 'percentile']).reset_index(drop=True)

percentiles_melted

,payload_date,tutorial_step_id,percentile,seconds
0,2026-04-01,app_launch,P10,0.194
1,2026-04-01,app_launch,P50,0.275
2,2026-04-01,app_launch,P90,0.558
3,2026-04-01,auth_process,P10,1.479
4,2026-04-01,auth_process,P50,2.343
...,...,...,...,...
1099,2026-05-05,service_manager_process,P50,0.764
1100,2026-05-05,service_manager_process,P90,1.845
1101,2026-05-05,update_app_popup,P10,4.629
1102,2026-05-05,update_app_popup,P50,10.452


In [50]:
data_bootstrap_agg = data_bootstrap.groupby(['payload_date', 'tutorial_step_id']).agg(
    unique_users = ('user_id', 'nunique'),
).reset_index()

data_bootstrap_agg

,payload_date,tutorial_step_id,unique_users
0,2026-04-01,app_launch,93889
1,2026-04-01,auth_process,93398
2,2026-04-01,display_applovin_consent,1665
3,2026-04-01,display_assets_download_popup,5277
4,2026-04-01,display_loading_screen,91970
...,...,...,...
363,2026-05-05,external_do_not_track_popup,69
364,2026-05-05,external_notifications_popup,1942
365,2026-05-05,helpshift_sdk,85712
366,2026-05-05,service_manager_process,85974


In [51]:
fig = px.line(data_bootstrap_agg, 
              x='payload_date', 
              y='unique_users',
              color='tutorial_step_id',
              title='Unique Users by Step Type and Payload Date',
              width=1200,
              height=600,
              hover_data={'unique_users': True, 'payload_date': True, 'tutorial_step_id': True},
)
fig.show()

In [53]:
fig = px.line(percentiles_melted, 
              x='payload_date', 
              y='seconds',
              color='tutorial_step_id',
              facet_row='percentile',
              title='Percentiles for step duration by step type and payload date',
              width=1200,
              height=1200,
              hover_data={'seconds': True, 'payload_date': True, 'tutorial_step_id': True},
)
fig.show()